# Cultist Wars: shared policy/value pilot
Enable GPU T4 ×2. Attach `nn-mcts-kaggle.zip` as a private Kaggle dataset. This notebook trains on verified expert replay outcomes, not MCTS self-play. Historical map splits are not a fresh strength holdout. Two-GPU path implemented but not GPU-tested locally. Small model may train faster on CPU due to GPU coordination overhead.


In [ ]:
from pathlib import Path
import zipfile, subprocess, sys, torch
bundle = next(Path('/kaggle/input').rglob('nn-mcts-kaggle.zip'))
work = Path('/kaggle/working/nn-mcts')
work.mkdir(exist_ok=False)
with zipfile.ZipFile(bundle) as z:
    z.extractall(work)
print(torch.__version__, 'CUDA devices:', torch.cuda.device_count())


In [ ]:
import hashlib, json
manifest = json.loads((work/'bundle-manifest.json').read_text())
for name, expected in manifest.items():
    assert hashlib.sha256((work/name).read_bytes()).hexdigest() == expected, name
print('Bundle checksums verified')


In [ ]:
# Batch size is per GPU. Fresh output directory required for each run.
args = ['train.py', '--cache', 'cache', '--out', 'trained', '--epochs', '30', '--batch-size', '512']
if torch.cuda.device_count() >= 2:
    command = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', *args]
else:
    command = [sys.executable, *args, '--device', 'cuda' if torch.cuda.is_available() else 'cpu']
subprocess.run(command, cwd=work, check=True)


In [ ]:
subprocess.run([sys.executable, 'export.py', '--checkpoint', 'trained/best.pt'], cwd=work, check=True)
print((work/'trained/report.json').read_text())
subprocess.run(['cc', '-O3', '-std=c11', 'test_mcts.c', '-lm', '-o', 'test_mcts'], cwd=work, check=True)
subprocess.run([str(work/'test_mcts')], cwd=work, check=True)
# Download output bundle; local parity and equal-time matches still required.
with zipfile.ZipFile('/kaggle/working/nn-mcts-results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ['trained/best.pt', 'trained/report.json', 'trained/history.json', 'candidate.c', 'packed.npy', 'export.json']:
        z.write(work/name, name)
